In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [1]:
import os
import random
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [2]:
BASE_PATH = "/kaggle/input/datasets/alekhya7gangopadhyay/eeg-dataset/content/drive/My Drive/Human Computer Interface (HCI)/Chebyshev Filtered Data"

SEQUENCE_LENGTH = 256

MAX_FILES_PER_CLASS = 500

RANDOM_STATE = 42

In [3]:
SELECTED_CHANNELS = [
    'P4 - O2',
    'P3 - O1',
    'F4 - C4'
]

In [4]:
label_map = {
    "Right": 0,
    "Left": 1,
    "Forward": 2,
    "Backward": 3
}

In [5]:
mandatory_files = {

    "Right": [
         "chebyshev_ARROW_Right.xlsx",
        "chebyshev_LETTER_Right.xlsx",
        "chebyshev_WORD_Right.xlsx",
        "chebyshev_Right ARROW2.xlsx",
        "chebyshev_Right LETTER2.xlsx",
        "chebyshev_Right WORD2.xlsx"
    ],

    "Left": [
        "chebyshev_ARROW_Left.xlsx",
        "chebyshev_LETTER_Left.xlsx",
        "chebyshev_WORD_Left.xlsx",
        "chebyshev_ARROW_Left_2.xlsx",
        "chebyshev_LETTER_Left_2.xlsx",
        "chebyshev_WORD_Left_2.xlsx"
    ],

    "Forward": [
       
        "chebyshev_ARROW_Forward.xlsx",
        "chebyshev_LETTER_Forward.xlsx",
        "chebyshev_WORD_Forward.xlsx",
        "chebyshev_ARROW_Forward_2.xlsx",
        "chebyshev_LETTER_Forward_2.xlsx",
        "chebyshev_WORD_Forward_2.xlsx"
    ],

    "Backward": [
       "chebyshev_ARROW_Backward.xlsx",
        "chebyshev_LETTER_Backward.xlsx",
        "chebyshev_WORD_Backword.xlsx",
        "chebyshev_ARROW_Backword_2.xlsx",
        "chebyshev_LETTER_Backward_2.xlsx",
        "chebyshev_WORD_Backward_2.xlsx"
    ]
}

In [6]:
random.seed(RANDOM_STATE)

selected_files = []

for class_name in label_map:

    folder = os.path.join(BASE_PATH, class_name)

    all_files = sorted([
        f for f in os.listdir(folder)
        if f.endswith(".xlsx")
    ])

    required = []

    for fname in mandatory_files[class_name]:

        if fname in all_files:
            required.append(fname)

    remaining = [
        f for f in all_files
        if f not in required
    ]

    needed = MAX_FILES_PER_CLASS - len(required)

    sampled = random.sample(
        remaining,
        needed
    )

    final_files = required + sampled

    print(
        f"{class_name}: "
        f"{len(final_files)} files selected "
        f"(mandatory={len(required)})"
    )

    for f in final_files:

        selected_files.append(
            (
                os.path.join(folder, f),
                label_map[class_name]
            )
        )

Right: 500 files selected (mandatory=6)
Left: 500 files selected (mandatory=6)
Forward: 500 files selected (mandatory=6)
Backward: 500 files selected (mandatory=6)


In [7]:
train_files = []
test_files = []

for class_name, label in label_map.items():

    class_files = [
        item
        for item in selected_files
        if item[1] == label
    ]

    train_cls, test_cls = train_test_split(
        class_files,
        test_size=0.2,
        random_state=RANDOM_STATE
    )

    train_files.extend(train_cls)
    test_files.extend(test_cls)

print()

print("Train files:", len(train_files))
print("Test files :", len(test_files))


Train files: 1600
Test files : 400


In [8]:
def create_sequences(data, seq_len):

    sequences = []

    for start in range(
        0,
        len(data) - seq_len + 1,
        seq_len
    ):

        sequences.append(
            data[start:start+seq_len]
        )

    return sequences

In [10]:
X_train = []
y_train = []

for idx, (filepath, label) in enumerate(train_files):

    if idx % 100 == 0:
        print(
            f"Processed {idx}/{len(train_files)} files"
        )

    try:

        df = pd.read_excel(filepath)

        df = df[SELECTED_CHANNELS]

        data = df.values.astype(np.float32)

        seqs = create_sequences(
            data,
            SEQUENCE_LENGTH
        )

        for seq in seqs:

            X_train.append(seq)

            y_train.append(label)

    except Exception as e:

        print(f"\nError in: {filepath}")
        print(e)

Processed 0/1600 files
Processed 100/1600 files
Processed 200/1600 files
Processed 300/1600 files
Processed 400/1600 files
Processed 500/1600 files
Processed 600/1600 files
Processed 700/1600 files
Processed 800/1600 files
Processed 900/1600 files
Processed 1000/1600 files
Processed 1100/1600 files
Processed 1200/1600 files
Processed 1300/1600 files
Processed 1400/1600 files
Processed 1500/1600 files


In [11]:
X_test = []
y_test = []

for filepath, label in test_files:

    try:

        df = pd.read_excel(filepath)

        df = df[SELECTED_CHANNELS]

        data = df.values.astype(np.float32)

        seqs = create_sequences(
            data,
            SEQUENCE_LENGTH
        )

        for seq in seqs:

            X_test.append(seq)

            y_test.append(label)

    except Exception as e:

        print(filepath)
        print(e)

X_test = np.array(
    X_test,
    dtype=np.float32
)

y_test = np.array(y_test)

print(X_test.shape)
print(y_test.shape)

(15840, 256, 3)
(15840,)


In [14]:
type(X_train)

list

In [15]:
X_train = np.array(
    X_train,
    dtype=np.float32
)

y_train = np.array(y_train)

print(X_train.shape)
print(y_train.shape)

(63544, 256, 3)
(63544,)


In [16]:
X_test = np.array(
    X_test,
    dtype=np.float32
)

y_test = np.array(y_test)

print(X_test.shape)
print(y_test.shape)

(15840, 256, 3)
(15840,)


In [17]:
channels = X_train.shape[2]

In [18]:
print("Normalizing...")

channels = X_train.shape[2]

scaler = StandardScaler()

X_train_2d = X_train.reshape(
    -1,
    channels
)

X_test_2d = X_test.reshape(
    -1,
    channels
)

X_train_2d = scaler.fit_transform(
    X_train_2d
)

X_test_2d = scaler.transform(
    X_test_2d
)

X_train = X_train_2d.reshape(
    X_train.shape
)

X_test = X_test_2d.reshape(
    X_test.shape
)

print("Normalization Complete")

print(X_train.shape)
print(X_test.shape)

Normalizing...
Normalization Complete
(63544, 256, 3)
(15840, 256, 3)


In [19]:
print(type(X_train))

if isinstance(X_train, list):
    print("Still a list")

else:
    print("NumPy array")
    print(X_train.shape)

<class 'numpy.ndarray'>
NumPy array
(63544, 256, 3)


In [20]:
type(X_train)
len(X_train)

63544

In [21]:
print(type(X_test))
print(X_test.shape)

print(type(y_test))
print(y_test.shape)

<class 'numpy.ndarray'>
(15840, 256, 3)
<class 'numpy.ndarray'>
(15840,)


In [22]:
from sklearn.preprocessing import StandardScaler

channels = X_train.shape[2]

scaler = StandardScaler()

X_train_2d = X_train.reshape(-1, channels)
X_test_2d  = X_test.reshape(-1, channels)

X_train_2d = scaler.fit_transform(X_train_2d)
X_test_2d  = scaler.transform(X_test_2d)

X_train = X_train_2d.reshape(X_train.shape)
X_test  = X_test_2d.reshape(X_test.shape)

print("Normalization complete")
print(X_train.shape)
print(X_test.shape)

Normalization complete
(63544, 256, 3)
(15840, 256, 3)


In [23]:
np.save("/kaggle/working/X_train.npy", X_train)
np.save("/kaggle/working/X_test.npy", X_test)
np.save("/kaggle/working/y_train.npy", y_train)
np.save("/kaggle/working/y_test.npy", y_test)

print("Saved successfully")

Saved successfully


In [24]:
import zipfile

with zipfile.ZipFile(
    "/kaggle/working/eeg_processed_dataset.zip",
    "w"
) as z:

    z.write("/kaggle/working/X_train.npy", "X_train.npy")
    z.write("/kaggle/working/X_test.npy", "X_test.npy")
    z.write("/kaggle/working/y_train.npy", "y_train.npy")
    z.write("/kaggle/working/y_test.npy", "y_test.npy")